# War Room Catalyst Review
**Charlie OS 5.0 | War Room v10.0**

Run all cells top to bottom to refresh. Each section is independent — jump to any one you need.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DB_PATH = Path("..") / "data" / "catalyst_tracker.db"

def query(sql, params=()):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return df

# Color map for conviction tags
CONVICTION_COLORS = {"HIGH": "#2ecc71", "DEVELOPING": "#f39c12", "WEAK": "#e74c3c"}
ENGINE_COLORS = {"F": "#3498db", "M": "#9b59b6", "N": "#e67e22",
                 "F+M": "#1abc9c", "F+N": "#2980b9", "ALL THREE": "#e74c3c"}

plt.style.use('dark_background')
print(f"Database: {DB_PATH.resolve()}")
print(f"Exists: {DB_PATH.exists()}")

---
## 1. Dashboard Summary

In [ ]:
totals = query("""
SELECT
    COUNT(*) as total_logged,
    SUM(CASE WHEN conviction_tag='HIGH' THEN 1 ELSE 0 END) as high,
    SUM(CASE WHEN conviction_tag='DEVELOPING' THEN 1 ELSE 0 END) as developing,
    SUM(CASE WHEN conviction_tag='WEAK' THEN 1 ELSE 0 END) as weak,
    SUM(is_earnings) as earnings_catalysts,
    COUNT(DISTINCT earnings_season) as seasons_tracked
FROM catalyst_log
""").iloc[0]

with_outcomes = query("""
SELECT COUNT(DISTINCT catalyst_id) as n FROM outcomes WHERE return_5d IS NOT NULL
""").iloc[0]['n']

seasons = query("""
SELECT DISTINCT earnings_season FROM catalyst_log
WHERE earnings_season IS NOT NULL ORDER BY earnings_season
""")['earnings_season'].tolist()

avg_score = query("SELECT AVG(score_total) as avg FROM catalyst_log").iloc[0]['avg']

print("=" * 50)
print("  WAR ROOM CATALYST TRACKER -- SUMMARY")
print("=" * 50)
print(f"  Total logged          : {int(totals.total_logged)}")
print(f"  With 5-day outcomes   : {int(with_outcomes)}")
print(f"  Avg score             : {avg_score:.1f}/10" if avg_score else "  Avg score             : n/a")
print(f"  HIGH conviction       : {int(totals.high)}")
print(f"  DEVELOPING            : {int(totals.developing)}")
print(f"  WEAK                  : {int(totals.weak)}")
print(f"  Earnings catalysts    : {int(totals.earnings_catalysts)}")
print(f"  Seasons tracked       : {', '.join(seasons) if seasons else 'none yet'}")

---
## 2. Full Catalyst Log

In [ ]:
df = query("""
SELECT
    cl.id,
    cl.analysis_date as date,
    cl.ticker,
    cl.score_total as score,
    cl.conviction_tag as conviction,
    cl.catalyst_type as type,
    cl.catalyst_engine as engine,
    cl.sector,
    cl.earnings_season as season,
    cl.macro_context as macro,
    cl.catalyst_direction as direction,
    cl.the_one_variable as one_variable,
    o.return_1d  as r1d,
    o.return_3d  as r3d,
    o.return_5d  as r5d,
    o.return_10d as r10d,
    o.return_20d as r20d,
    o.catalyst_resolved as resolved
FROM catalyst_log cl
LEFT JOIN outcomes o ON o.catalyst_id = cl.id
ORDER BY cl.analysis_date DESC, cl.id DESC
""")

def color_score(val):
    if pd.isna(val): return ''
    if val >= 8: return 'background-color: #1a4a2e; color: #2ecc71'
    if val >= 5: return 'background-color: #4a3a1a; color: #f39c12'
    return 'background-color: #4a1a1a; color: #e74c3c'

def color_return(val):
    if pd.isna(val): return ''
    return 'color: #2ecc71' if val > 0 else 'color: #e74c3c'

styled = (
    df.style
    .map(color_score, subset=['score'])
    .map(color_return, subset=['r1d','r3d','r5d','r10d','r20d'])
    .format({
        'score': '{:.1f}',
        'r1d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r3d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r5d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r10d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r20d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
    })
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#1a1a2e'), ('color', '#aaa')]}])
)

display(styled)

---
## 3. Earnings Season Scoreboard

In [ ]:
# Change SEASON to focus on a specific one, or leave None for latest
SEASON = None  # e.g. "Q1-2026"

if SEASON is None:
    latest = query("""
    SELECT earnings_season FROM catalyst_log
    WHERE is_earnings=1 ORDER BY analysis_date DESC LIMIT 1
    """)
    SEASON = latest.iloc[0]['earnings_season'] if not latest.empty else None

if SEASON:
    es = query("""
    SELECT cl.ticker, cl.analysis_date as date, cl.score_total as score,
           cl.conviction_tag as conviction, cl.catalyst_type_label as type,
           cl.catalyst_engine as engine, cl.macro_context as macro,
           o.return_1d as r1d, o.return_3d as r3d,
           o.return_5d as r5d, o.return_10d as r10d,
           o.catalyst_resolved as resolved
    FROM catalyst_log cl
    LEFT JOIN outcomes o ON o.catalyst_id = cl.id
    WHERE cl.earnings_season = ? AND cl.is_earnings = 1
    ORDER BY cl.score_total DESC
    """, params=(SEASON,))

    print(f"Earnings Season: {SEASON}  |  {len(es)} catalysts")
    if not es.empty:
        win_rate = (es['r5d'] > 0).sum() / es['r5d'].notna().sum() * 100 if es['r5d'].notna().sum() > 0 else None
        avg_5d = es['r5d'].mean()
        print(f"Avg score: {es['score'].mean():.1f} | Avg 5d return: {avg_5d:+.2f}%" if pd.notna(avg_5d) else f"Avg score: {es['score'].mean():.1f} | 5d returns: pending")
        if win_rate is not None:
            print(f"5d win rate: {win_rate:.0f}%")

    styled_es = (
        es.style
        .map(color_score, subset=['score'])
        .map(color_return, subset=['r1d','r3d','r5d','r10d'])
        .format({
            'score': '{:.1f}',
            'r1d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r3d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r5d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r10d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        })
    )
    display(styled_es)
else:
    print("No earnings catalysts logged yet.")

---
## 4. Performance Charts

In [ ]:
perf = query("""
SELECT cl.conviction_tag, cl.catalyst_engine,
       o.return_1d, o.return_3d, o.return_5d, o.return_10d
FROM catalyst_log cl
JOIN outcomes o ON o.catalyst_id = cl.id
WHERE o.return_5d IS NOT NULL
""")

if perf.empty:
    print("No outcome data yet. Log some outcomes first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0d0d0d')

    # -- Chart 1: Avg return by conviction tier --
    ax = axes[0]
    ax.set_facecolor('#1a1a1a')
    tiers = ['HIGH', 'DEVELOPING', 'WEAK']
    windows = ['return_1d', 'return_3d', 'return_5d', 'return_10d']
    labels = ['1d', '3d', '5d', '10d']
    x = range(len(labels))
    for tier in tiers:
        subset = perf[perf['conviction_tag'] == tier]
        if not subset.empty:
            avgs = [subset[w].mean() for w in windows]
            ax.plot(labels, avgs, marker='o', label=tier,
                    color=CONVICTION_COLORS.get(tier, 'white'), linewidth=2)
    ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.set_title('Avg Return by Conviction Tier', color='white')
    ax.set_ylabel('Return %', color='white')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#1a1a1a', labelcolor='white')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))

    # -- Chart 2: Avg 5d return by catalyst engine --
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a1a')
    eng_data = perf.groupby('catalyst_engine')['return_5d'].agg(['mean', 'count']).reset_index()
    eng_data = eng_data.sort_values('mean', ascending=False)
    colors = [ENGINE_COLORS.get(e, '#888') for e in eng_data['catalyst_engine']]
    bars = ax2.bar(eng_data['catalyst_engine'], eng_data['mean'], color=colors, alpha=0.85)
    ax2.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    for bar, count in zip(bars, eng_data['count']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'n={count}', ha='center', va='bottom', color='#aaa', fontsize=8)
    ax2.set_title('Avg 5d Return by Engine', color='white')
    ax2.set_ylabel('5d Return %', color='white')
    ax2.tick_params(colors='#aaa')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))

    # -- Chart 3: Score distribution histogram --
    ax3 = axes[2]
    ax3.set_facecolor('#1a1a1a')
    scores = query("SELECT score_total FROM catalyst_log")['score_total'].dropna()
    ax3.hist(scores, bins=20, color='#3498db', alpha=0.8, edgecolor='#1a1a1a')
    ax3.axvline(8, color='#2ecc71', linewidth=1.5, linestyle='--', label='HIGH threshold')
    ax3.axvline(5, color='#f39c12', linewidth=1.5, linestyle='--', label='DEVELOPING threshold')
    ax3.set_title('Score Distribution', color='white')
    ax3.set_xlabel('Catalyst Score', color='white')
    ax3.set_ylabel('Count', color='white')
    ax3.tick_params(colors='#aaa')
    ax3.legend(facecolor='#1a1a1a', labelcolor='white')

    plt.tight_layout()
    plt.show()

---
## 5. Performance by Catalyst Type

In [ ]:
by_type = query("""
SELECT cl.catalyst_type, cl.catalyst_type_label,
       COUNT(*) as n,
       ROUND(AVG(cl.score_total), 2) as avg_score,
       ROUND(AVG(o.return_1d), 2)  as avg_1d,
       ROUND(AVG(o.return_3d), 2)  as avg_3d,
       ROUND(AVG(o.return_5d), 2)  as avg_5d,
       ROUND(AVG(o.return_10d), 2) as avg_10d,
       SUM(CASE WHEN o.return_5d > 0 THEN 1 ELSE 0 END) as wins_5d,
       SUM(CASE WHEN o.return_5d IS NOT NULL THEN 1 ELSE 0 END) as with_outcome
FROM catalyst_log cl
LEFT JOIN outcomes o ON o.catalyst_id = cl.id
WHERE cl.catalyst_type IS NOT NULL
GROUP BY cl.catalyst_type
ORDER BY avg_5d DESC
""")

if by_type.empty:
    print("No data yet.")
else:
    by_type['win_rate_5d'] = by_type.apply(
        lambda r: f"{r['wins_5d']/r['with_outcome']*100:.0f}%" if r['with_outcome'] > 0 else '-', axis=1
    )
    display_cols = ['catalyst_type', 'catalyst_type_label', 'n', 'avg_score',
                    'avg_1d', 'avg_3d', 'avg_5d', 'avg_10d', 'win_rate_5d']
    styled_type = (
        by_type[display_cols].style
        .map(color_return, subset=['avg_1d','avg_3d','avg_5d','avg_10d'])
        .format({
            'avg_score': '{:.1f}',
            'avg_1d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'avg_3d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'avg_5d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'avg_10d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        })
    )
    display(styled_type)

---
## 6. Ticker Lookup
Change `TICKER` to any symbol you've logged.

In [ ]:
TICKER = "NVDA"  # <-- change this

tk = query("""
SELECT cl.id, cl.analysis_date as date, cl.score_total as score,
       cl.conviction_tag as conviction, cl.catalyst_type_label as type,
       cl.catalyst_engine as engine, cl.earnings_season as season,
       cl.macro_context as macro, cl.catalyst_direction as direction,
       cl.the_one_variable as one_variable,
       cl.thesis_summary as thesis,
       o.price_at_analysis as entry_price,
       o.return_1d as r1d, o.return_3d as r3d,
       o.return_5d as r5d, o.return_10d as r10d, o.return_20d as r20d,
       o.catalyst_resolved as resolved, o.resolution_notes as notes
FROM catalyst_log cl
LEFT JOIN outcomes o ON o.catalyst_id = cl.id
WHERE cl.ticker = ?
ORDER BY cl.analysis_date DESC
""", params=(TICKER.upper(),))

if tk.empty:
    print(f"No entries found for {TICKER.upper()}")
else:
    print(f"{TICKER.upper()} -- {len(tk)} entries")
    for _, row in tk.iterrows():
        print(f"\n  ID #{int(row['id'])} | {row['date']} | Score: {row['score']:.1f}/10 | {row['conviction']}")
        print(f"  Engine: {row['engine']} | Type: {str(row['type'])[:45]}")
        if pd.notna(row['one_variable']):
            print(f"  One Variable: {row['one_variable']}")
        if pd.notna(row['thesis']):
            print(f"  Thesis: {row['thesis']}")
        r_parts = []
        for label, col in [('1d', 'r1d'), ('3d', 'r3d'), ('5d', 'r5d'), ('10d', 'r10d'), ('20d', 'r20d')]:
            r_parts.append(f"{label}: {row[col]:+.2f}%" if pd.notna(row[col]) else f"{label}: -")
        print(f"  Returns -> {' | '.join(r_parts)}")
        if pd.notna(row['resolved']):
            print(f"  Resolved: {row['resolved']} -- {row['notes'] or ''}")

---
## 7. Score vs 5-Day Return Scatter

In [ ]:
scatter = query("""
SELECT cl.score_total, cl.conviction_tag, cl.catalyst_engine,
       cl.ticker, o.return_5d
FROM catalyst_log cl
JOIN outcomes o ON o.catalyst_id = cl.id
WHERE o.return_5d IS NOT NULL
""")

if scatter.empty:
    print("No outcome data yet.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#0d0d0d')
    ax.set_facecolor('#1a1a1a')

    for tag in ['HIGH', 'DEVELOPING', 'WEAK']:
        subset = scatter[scatter['conviction_tag'] == tag]
        if not subset.empty:
            ax.scatter(subset['score_total'], subset['return_5d'],
                      label=tag, color=CONVICTION_COLORS.get(tag, 'white'),
                      s=80, alpha=0.85, edgecolors='none')
            for _, row in subset.iterrows():
                ax.annotate(row['ticker'], (row['score_total'], row['return_5d']),
                           textcoords='offset points', xytext=(5, 3),
                           fontsize=7, color='#aaa')

    ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.axvline(8, color='#2ecc71', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(5, color='#f39c12', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('Catalyst Score', color='white')
    ax.set_ylabel('5-Day Return %', color='white')
    ax.set_title('Score vs 5-Day Return (catalyst direction)', color='white')
    ax.tick_params(colors='#aaa')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))
    ax.legend(facecolor='#1a1a1a', labelcolor='white')
    plt.tight_layout()
    plt.show()

---
## 8. Pending Outcomes
Catalysts that have been logged but still need price follow-through.

In [ ]:
pending = query("""
SELECT cl.id, cl.analysis_date as date, cl.ticker,
       cl.score_total as score, cl.conviction_tag as conviction,
       cl.catalyst_engine as engine, cl.catalyst_direction as direction,
       o.price_at_analysis as entry,
       o.return_1d as r1d, o.return_3d as r3d,
       o.return_5d as r5d, o.return_10d as r10d
FROM catalyst_log cl
LEFT JOIN outcomes o ON o.catalyst_id = cl.id
WHERE o.return_5d IS NULL
ORDER BY cl.analysis_date DESC
""")

if pending.empty:
    print("All logged catalysts have 5-day outcomes. Nothing pending.")
else:
    print(f"{len(pending)} catalysts pending outcome data")
    print("Tell Claude: 'log the outcome for [TICKER]' with the prices\n")
    display(pending.style
        .map(color_score, subset=['score'])
        .format({'score': '{:.1f}',
                 'entry': lambda x: f"${x:.2f}" if pd.notna(x) else 'pending'}))